# Training: Predicting Negative Electricity Prices

**Goal:** predict from the weather forecast whether an hour on the Austrian day-ahead market will have a price ≤ 0 €/MWh.

**Data:** read directly from Kafka (offset 0, batch mode):
- `awattar.prices`: hourly day-ahead prices (label)
- `openmeteo.historical_forecast`: what the weather models *predicted* back then, for 10 AT/DE locations (features)

We train on historical *forecasts*, not on measured weather, because the live model will also only see forecasts. Training on measured weather would make the model look better than it really is.

**Models compared:**
1. Logistic regression baseline (all features as plain numbers)
2. Improved logistic regression (one-hot hour/month, class weights, DE solar/wind totals)
3. Gradient-boosted trees (same features, can learn interactions on its own)

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, OneHotEncoder
from pyspark.ml.classification import LogisticRegression, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array

spark = (SparkSession.builder.appName("train-cheap-hours")
         .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")   # hide Spark's harmless WARN messages

# Dev container sets KAFKA_BOOTSTRAP=kafka:9092; on the host the default is localhost:9094
BOOT = os.getenv("KAFKA_BOOTSTRAP", "localhost:9094")
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))   # so `import src...` works from notebooks/

from src.features import (CALENDAR, PRICE_SCHEMA, WEATHER_SCHEMA,
                          numeric_feature_cols, parse_kafka, weather_features)


def read_topic(topic, schema):
    """Batch-read a whole Kafka topic (offset 0 -> latest) and parse the JSON values."""
    raw = (spark.read.format("kafka").option("kafka.bootstrap.servers", BOOT)
           .option("subscribe", topic).option("startingOffsets", "earliest").load())
    return parse_kafka(raw, schema)

print("Kafka:", BOOT)

## 2. Features

`weather_features()` turns the long weather table (one row per location and hour) into one row per hour:

- **Per location:** solar radiation, wind at 100 m, temperature, cloud cover (e.g. `munich_rad`, `husum_wind`)
- **DE totals:** prices react to *total* German solar and wind output, not to a single city, so we add `de_solar_sum` and `de_wind_mean`
- **Calendar:** hour of day and month (local time Europe/Vienna) and a weekend flag

The function lives in [`src/features.py`](../src/features.py). The live scoring job (`src/predict.py`) imports **the same function**, so the model always sees the same columns as in training.

In [ ]:
import inspect
print(inspect.getsource(weather_features))   # defined in src/features.py

## 3. Data, label and split

- **Label:** `1` if the price is ≤ 0 €/MWh, else `0`
- **Split by time, not randomly:** neighbouring hours are almost identical, so a random split would leak test information into training.
- **Training starts in 2023:** in 2022 (energy crisis) negative prices almost never happened (0.02 % of hours). Since then their share has roughly doubled every year. Including 2022 teaches the model that negative prices "basically never happen".

In [ ]:
prices = (read_topic("awattar.prices", PRICE_SCHEMA)
          .groupBy("ts").agg(F.max_by("price_eur_mwh", "ingested_at").alias("price")))
X = weather_features(read_topic("openmeteo.historical_forecast", WEATHER_SCHEMA))

data = (X.join(prices, "ts")
         .withColumn("label", (F.col("price") <= 0).cast("double"))
         .cache())   # read Kafka once, reuse for all models

print("Share of negative-price hours per year:")
(data.groupBy(F.year("ts").alias("year"))
     .agg(F.count("*").alias("hours"), F.round(F.avg("label") * 100, 2).alias("negative_%"))
     .orderBy("year").show())

TRAIN_START, TEST_START = "2023-01-01", "2025-07-01"
train = data.filter((F.col("ts") >= TRAIN_START) & (F.col("ts") < TEST_START))
test = data.filter(F.col("ts") >= TEST_START)

# Class weights: both classes count equally in the loss, despite ~4 % positives
pos = train.filter("label = 1").count() / train.count()
train = train.withColumn("w", F.when(F.col("label") == 1, 0.5 / pos).otherwise(0.5 / (1 - pos)))
print(f"train: {train.count()} hours ({pos:.1%} negative)   test: {test.count()} hours")

## 4. Evaluation helper

Negative-price hours are rare, so **accuracy and ROC-AUC look good even for a weak model**. We compare models mainly on:

- **PR-AUC:** quality of the ranking for the rare class (random guessing ≈ share of negative hours)
- **Precision:** when the model says "price ≤ 0", how often is it right?
- **Recall:** how many of the negative-price hours does it catch?

In [ ]:
THRESHOLDS = [0.2, 0.3, 0.5, 0.7]


def evaluate(model, df, name):
    pred = model.transform(df).withColumn("p", vector_to_array("probability")[1]).cache()
    row = {"model": name,
           "PR-AUC": BinaryClassificationEvaluator(metricName="areaUnderPR").evaluate(pred),
           "ROC-AUC": BinaryClassificationEvaluator(metricName="areaUnderROC").evaluate(pred)}
    pos_label = F.col("label") == 1
    counts = pred.agg(*[
        agg for t in THRESHOLDS for agg in (
            F.sum(((F.col("p") >= t) & pos_label).cast("int")).alias(f"tp_{t}"),
            F.sum(((F.col("p") >= t) & ~pos_label).cast("int")).alias(f"fp_{t}"),
            F.sum(((F.col("p") < t) & pos_label).cast("int")).alias(f"fn_{t}"))
    ]).first()
    for t in THRESHOLDS:
        tp, fp, fn = counts[f"tp_{t}"], counts[f"fp_{t}"], counts[f"fn_{t}"]
        row[f"precision@{t}"] = tp / max(tp + fp, 1)
        row[f"recall@{t}"] = tp / max(tp + fn, 1)
    pred.unpersist()
    return row


results = []

## 5. Model 1: Logistic regression baseline

All features as plain numbers, no class weights. Problem: `hour` and `month` enter as straight lines ("later hour = more/less likely"), but negative prices peak around **midday** and in **spring/summer**.

In [ ]:
numeric_cols = numeric_feature_cols(X)

baseline = Pipeline(stages=[
    VectorAssembler(inputCols=numeric_cols + CALENDAR, outputCol="raw"),
    StandardScaler(inputCol="raw", outputCol="features"),
    LogisticRegression(featuresCol="features", labelCol="label", regParam=0.01),
]).fit(train)
results.append(evaluate(baseline, test, "1 LR baseline"))
pd.DataFrame(results).round(3)

## 6. Model 2: Improved logistic regression

- **One-hot hour and month:** each hour/month gets its own weight, so the midday peak and the seasons can be learned
- **Class weights** (`w`): stop the model from always predicting "no" for the rare class
- **DE solar/wind totals** from the feature function

In [ ]:
onehot = OneHotEncoder(inputCols=CALENDAR, outputCols=["hour_oh", "month_oh"])
assemble_num = VectorAssembler(inputCols=numeric_cols, outputCol="num")
scale = StandardScaler(inputCol="num", outputCol="num_scaled")
assemble_all = VectorAssembler(inputCols=["num_scaled", "hour_oh", "month_oh"], outputCol="features")

lr_improved = Pipeline(stages=[
    onehot, assemble_num, scale, assemble_all,
    LogisticRegression(featuresCol="features", labelCol="label", weightCol="w", regParam=0.01),
]).fit(train)
results.append(evaluate(lr_improved, test, "2 LR improved"))
pd.DataFrame(results).round(3)

## 7. Model 3: Gradient-boosted trees

Same features and class weights. Trees can learn combinations such as "sunny **and** weekend **and** midday" by themselves, which a linear model can't.

In [ ]:
gbt = Pipeline(stages=[
    onehot, assemble_num, scale, assemble_all,
    GBTClassifier(featuresCol="features", labelCol="label", weightCol="w", maxIter=50, maxDepth=5, seed=42),
]).fit(train)
results.append(evaluate(gbt, test, "3 GBT"))
pd.DataFrame(results).round(3)

## 8. Comparison and saving the best model

The best model by PR-AUC is saved to `models/cheap_hours`; the live scoring job (`python -m src.predict`) loads it from there.

**Choosing the threshold:** for EV charging or a heat pump, a false alarm costs little, but a missed free hour is lost savings. So a threshold of about 0.3 is usually more useful than the default 0.5.

In [ ]:
comparison = pd.DataFrame(results).set_index("model").round(3)
display(comparison)

models = {"1 LR baseline": baseline, "2 LR improved": lr_improved, "3 GBT": gbt}
best_name = comparison["PR-AUC"].idxmax()
models[best_name].write().overwrite().save(str(PROJECT_ROOT / "models" / "cheap_hours"))
print(f"Saved best model: {best_name} -> models/cheap_hours")

## 9. What drives negative prices?

Coefficients of the improved logistic regression. The numeric features are standardized, so their weights are directly comparable. Positive = makes a negative price more likely.

In [ ]:
lr_stage = lr_improved.stages[-1]
weights = lr_stage.coefficients.toArray()[:len(numeric_cols)]   # first part of the vector = numeric features
coef = (pd.DataFrame({"feature": numeric_cols, "weight": weights})
          .sort_values("weight", key=abs, ascending=False).head(15).reset_index(drop=True))
coef.round(3)